<a href="https://colab.research.google.com/github/Zsuz-devon/PRE_HACKIATON/blob/main/Codigo_HackIAton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install -q crewai litellm langchain-groq streamlit

In [17]:
import json

polizas = {
  "pacientes": [
    {"cedula": "0912345678", "nombre": "Ana López", "plan": "Premium Health", "cobertura_porcentaje": 80, "copago_fijo_emergencia": 20.00, "carencia_activa": False},
    {"cedula": "0987654321", "nombre": "Carlos Mendoza", "plan": "Salud Básica", "cobertura_porcentaje": 60, "copago_fijo_emergencia": 35.00, "carencia_activa": False}
  ]
}

red_hosp = {
  "hospitales": [
    {"id_hospital": "H01", "nombre": "Hospital San Juan", "especialidades": ["Traumatología", "Cardiología", "Medicina General"], "tarifa_base_consulta": 80.00, "tiempo_espera_minutos": 15},
    {"id_hospital": "H02", "nombre": "Clínica del Sur", "especialidades": ["Gastroenterología", "Pediatría", "Medicina General"], "tarifa_base_consulta": 50.00, "tiempo_espera_minutos": 40}
  ]
}

with open('polizas_seguro.json', 'w', encoding='utf-8') as f: json.dump(polizas, f, ensure_ascii=False, indent=2)
with open('red_hospitalaria.json', 'w', encoding='utf-8') as f: json.dump(red_hosp, f, ensure_ascii=False, indent=2)

In [18]:
%%writefile app.py
import streamlit as st
import os
import json
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

# --- SOLUCIÓN: PARCHE PARA EL BUG DE CREWAI + GROQ ---
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg
# -----------------------------------------------------

# Configuramos la página de la aplicación
st.set_page_config(page_title="Estimador de Copago V1.0", page_icon="🏥", layout="wide")

# ... (todo el resto de tu código se mantiene exactamente igual de aquí hacia abajo)

with st.sidebar:
    st.header("⚙️ Configuración")
    api_key_input = st.text_input("API Key de Groq:", type="password", help="Ingresa tu token de consola de Groq.")
    st.markdown("---")
    st.markdown("**¿Qué hace este sistema?**\nAnaliza tus síntomas, busca el mejor hospital en la red y calcula tu copago exacto basado en tu póliza.")

st.title("Asistente Inteligente de Triaje y Seguros")

col1, col2 = st.columns(2)
with col1:
    cedula_paciente = st.text_input("Número de Cédula (Ej: 0912345678):")
with col2:
    sintomas = st.text_input("¿Qué síntomas presentas hoy?")

if st.button("Consultar Cobertura y Hospital", type="primary"):
    if not api_key_input:
        st.error("⚠️ Por favor, ingresa tu API Key de Groq en la barra lateral antes de continuar.")
        st.stop()

    if not cedula_paciente or not sintomas:
        st.warning("⚠️ Por favor, ingresa tanto la cédula como los síntomas para poder ayudarte.")
        st.stop()

    with st.spinner('Activando agentes médicos y evaluando póliza...'):
        try:
            # 1. Configuración del LLM nativa para Groq vía LiteLLM
            cerebro_groq = LLM(
                model="groq/llama-3.3-70b-versatile",
                api_key=api_key_input,
                temperature=0.1 # Temperatura baja para cálculos matemáticos precisos
            )

            # 2. Herramientas
            @tool("consultar_poliza")
            def consultar_poliza(cedula: str) -> str:
                """Consulta los datos de cobertura y copago de un paciente usando su cédula."""
                try:
                    with open('polizas_seguro.json', 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        paciente = next((p for p in data['pacientes'] if p['cedula'] == cedula), None)
                        return json.dumps(paciente, ensure_ascii=False) if paciente else "Error: Paciente no encontrado."
                except Exception as e:
                    return f"Error interno al leer pólizas: {str(e)}"

            @tool("consultar_red_hospitalaria")
            def consultar_red_hospitalaria(consulta: str = "todo") -> str:
                """Devuelve la lista de hospitales, especialidades y tarifas base."""
                try:
                    with open('red_hospitalaria.json', 'r', encoding='utf-8') as f:
                        return json.dumps(json.load(f), ensure_ascii=False)
                except Exception as e:
                    return f"Error interno al leer red hospitalaria: {str(e)}"

            # 3. Agentes
            agente_auditor = Agent(
                role='Auditor de Pólizas',
                goal='Extraer el porcentaje de cobertura y datos del plan del paciente usando su cédula.',
                backstory="Eres un auditor estricto de una aseguradora. Solo reportas la información exacta de la póliza.",
                llm=cerebro_groq,
                tools=[consultar_poliza],
                verbose=False
            )

            agente_ruteador = Agent(
                role='Ruteador Médico',
                goal='Analizar el síntoma, determinar la especialidad médica requerida y buscar el hospital más económico que la ofrezca.',
                backstory="Eres un experto en triaje médico. Cruzas los síntomas del paciente con las capacidades y tarifas de los hospitales disponibles.",
                llm=cerebro_groq,
                tools=[consultar_red_hospitalaria],
                verbose=False
            )

            agente_asesor = Agent(
                role='Asesor de Admisiones',
                goal='Calcular el valor final a pagar y redactar una respuesta empática para el paciente.',
                backstory="Eres el contacto final con el paciente. Tomas las tarifas del hospital, aplicas el porcentaje de cobertura del seguro, calculas el copago y le explicas amablemente a dónde ir.",
                llm=cerebro_groq,
                verbose=False
            )

            # 4. Tareas
            tarea_poliza = Task(
                description=f"Busca la póliza del paciente con cédula: {cedula_paciente}. Extrae su plan y porcentaje de cobertura.",
                expected_output="Un reporte corto con el nombre del paciente y su porcentaje de cobertura. Si no existe, indícalo claramente.",
                agent=agente_auditor
            )

            tarea_hospital = Task(
                description=f"El paciente reporta: '{sintomas}'. Determina la especialidad requerida y extrae el nombre del hospital adecuado y su tarifa base.",
                expected_output="Nombre del hospital recomendado, especialidad sugerida y la tarifa base de la consulta.",
                agent=agente_ruteador
            )

            tarea_calculo_final = Task(
                description="""
                Usa los resultados anteriores.
                1. Calcula el Copago: Copago = Tarifa Base - (Tarifa Base * (Porcentaje Cobertura / 100)).
                2. Redacta un mensaje amable al paciente indicando a qué hospital dirigirse, a qué especialidad, y el valor exacto de su copago.
                """,
                expected_output="Mensaje final empático dirigido al paciente con el cálculo matemático exacto y las instrucciones de a dónde ir.",
                agent=agente_asesor,
                context=[tarea_poliza, tarea_hospital]
            )

            # 5. Ejecución
            tripulacion_salud = Crew(
                agents=[agente_auditor, agente_ruteador, agente_asesor],
                tasks=[tarea_poliza, tarea_hospital, tarea_calculo_final],
                process=Process.sequential,
                verbose=False
            )

            resultado = tripulacion_salud.kickoff()

            st.success("¡Análisis completado!")
            st.info(resultado.raw)

        except Exception as e:
            st.error(f"Se produjo un error en la ejecución de los agentes: {str(e)}")

Overwriting app.py


In [19]:
import os
import time
import urllib.request

print("🧹 Limpiando procesos antiguos...")
os.system("pkill -f streamlit")
os.system("pkill -f localtunnel")
time.sleep(2)

print("🚀 Iniciando Streamlit en segundo plano...")
os.system("nohup streamlit run app.py --server.port 8501 > app.log 2>&1 &")

# Esperamos a que Streamlit cargue correctamente
time.sleep(5)

# Localtunnel requiere la IP de tu servidor de Colab como contraseña
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("="*50)
print(f"🔑 ATENCIÓN: COPIA ESTA IP, SERÁ TU CONTRASEÑA: {ip}")
print("="*50)

print("\n🌐 Generando tu enlace público (haz clic en el link que termine en '.loca.lt')...")
!npx localtunnel --port 8501

🧹 Limpiando procesos antiguos...
🚀 Iniciando Streamlit en segundo plano...
🔑 ATENCIÓN: COPIA ESTA IP, SERÁ TU CONTRASEÑA: 107.178.216.85

🌐 Generando tu enlace público (haz clic en el link que termine en '.loca.lt')...
⠙⠹⠸⠼⠴your url is: https://tasty-lights-build.loca.lt
^C
